# TCC2 — Dengue Forecasting: Model Training (Distrito Federal)

XGBoost training with hyperparameter optimization, comparing regression and classification approaches for t+4 week dengue notification forecasting.

**Author:** Pedro Santana — Engenharia de Software, Universidade de Brasilia
**Notebook 2 of 3:** Data Preparation → **Model Training** → Explainability (SHAP)

---

## Overview

This notebook runs **7 experiments** to find the best dengue forecasting model for Distrito Federal (IBGE 5300108):

| # | Experiment | Type | Goal |
|---|-----------|------|------|
| 1 | Baseline Regression | Regression | Performance floor |
| 2 | Optuna Regression Tuning | Tuning | Hyperparameter search (50 trials) |
| 3 | Tuned Regression | Regression | Best regression model |
| 4 | SINAN-only vs SINAN+INMET | Ablation | Value of meteorological data |
| 5 | Baseline Classification | Classification | Risk-level performance floor |
| 6 | Optuna Classification Tuning | Tuning | Hyperparameter search (50 trials) |
| 7 | Tuned Classification | Classification | Best classification model |

Plus **walk-forward cross-validation** to assess temporal stability.

### Critical Context: Distribution Shift
2024 was a historically unprecedented dengue year in Brasilia. Training max = 3,968 notifications vs test max = 22,278 (a **5.6x gap**). We apply `log1p` transform to the regression target, which compresses this gap to ~1.2x in log space.

## 1. Environment Setup

In [ ]:
%%capture
!pip install -q xgboost optuna shap huggingface_hub datasets scikit-learn matplotlib seaborn

In [ ]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import optuna
from optuna.pruners import MedianPruner
from optuna.visualization import plot_optimization_history, plot_param_importances

from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})

# ── Paths ──
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── HuggingFace config ──
HF_REPO = "pedrolucassantanaf/dengue-tcc2-data"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Output directory : {OUTPUT_DIR}")
print(f"HF repo          : {HF_REPO}")
print(f"HF token loaded  : {HF_TOKEN is not None}")
print(f"XGBoost version  : {xgb.__version__}")
print(f"Optuna version   : {optuna.__version__}")

## 2. Load and Prepare Data

In [ ]:
# ── Download splits from HuggingFace ──
def load_split(split_name: str) -> pd.DataFrame:
    """Download a parquet split from HuggingFace and return as DataFrame."""
    path = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"data/model_ready/{split_name}.parquet",
        repo_type="dataset",
        token=HF_TOKEN,
    )
    return pd.read_parquet(path)

t0 = time.time()
df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")
print(f"Downloaded in {time.time() - t0:.1f}s")

# ── Filter to Distrito Federal (IBGE 5300108) ──
DF_CODE = "5300108"
df_train = df_train[df_train["ibge_municipio"] == DF_CODE].copy()
df_val   = df_val[df_val["ibge_municipio"] == DF_CODE].copy()
df_test  = df_test[df_test["ibge_municipio"] == DF_CODE].copy()

print(f"\nAfter filtering to DF ({DF_CODE}):")
print(f"  Train : {df_train.shape}")
print(f"  Val   : {df_val.shape}")
print(f"  Test  : {df_test.shape}")

In [ ]:
# ── Combine train + val for final training ──
df_trainval = pd.concat([df_train, df_val], ignore_index=True)
print(f"Combined train+val : {df_trainval.shape}")

# ── Column definitions ──
META_COLS = ["ibge_municipio", "ano", "semana_epidemiologica"]
TARGET_REG = "notificacoes_t4"
TARGET_CLS = "risco_surto_t4"

# INMET meteorological columns (for SINAN-only ablation)
INMET_COLS = [
    "rain_sum_mm", "rain_mean_mm", "temp_mean_c", "temp_min_c", "temp_max_c",
    "humidity_mean_pct", "pressure_mean_mbar", "wind_speed_mean_ms",
    "radiation_mean_kj", "n_valid_hours", "rain_days", "rain_heavy_days",
    "temp_range_c", "low_coverage",
    # Lags
    "rain_sum_mm_lag_1", "rain_sum_mm_lag_2", "rain_sum_mm_lag_4", "rain_sum_mm_lag_8",
    "temp_mean_c_lag_1", "temp_mean_c_lag_2", "temp_mean_c_lag_4", "temp_mean_c_lag_8",
    "humidity_mean_pct_lag_1", "humidity_mean_pct_lag_2",
    "humidity_mean_pct_lag_4", "humidity_mean_pct_lag_8",
    # Moving averages
    "rain_sum_mm_mm4", "temp_mean_c_mm4", "humidity_mean_pct_mm4",
    # Additional lags
    "rain_heavy_days_lag2", "rain_heavy_days_lag4", "temp_range_c_lag2",
]

# Verify INMET columns exist in the data
inmet_present = [c for c in INMET_COLS if c in df_trainval.columns]
inmet_missing = [c for c in INMET_COLS if c not in df_trainval.columns]
print(f"\nINMET columns present : {len(inmet_present)}/{len(INMET_COLS)}")
if inmet_missing:
    print(f"  Missing: {inmet_missing}")
    INMET_COLS = inmet_present  # use only what exists

In [ ]:
def split_xy(df, target, drop_inmet=False):
    """
    Split DataFrame into features (X) and target (y).

    Drops META_COLS and both target columns from features.
    If drop_inmet=True, also drops INMET meteorological columns.
    """
    drop_cols = META_COLS + [TARGET_REG, TARGET_CLS]
    if drop_inmet:
        drop_cols = drop_cols + INMET_COLS

    # Only drop columns that exist
    drop_cols = [c for c in drop_cols if c in df.columns]

    X = df.drop(columns=drop_cols)
    y = df[target].values
    return X, y

# ── Prepare splits ──
X_trainval, y_reg_trainval = split_xy(df_trainval, TARGET_REG)
X_test, y_reg_test         = split_xy(df_test, TARGET_REG)

_, y_cls_trainval = split_xy(df_trainval, TARGET_CLS)
_, y_cls_test     = split_xy(df_test, TARGET_CLS)

# ── Log-transform regression target ──
y_reg_trainval_log = np.log1p(y_reg_trainval)
y_reg_test_log     = np.log1p(y_reg_test)

feature_names = list(X_trainval.columns)

print(f"Features         : {len(feature_names)}")
print(f"X_trainval shape : {X_trainval.shape}")
print(f"X_test shape     : {X_test.shape}")
print(f"\nRegression target (original scale):")
print(f"  Train+val range: [{y_reg_trainval.min():.0f}, {y_reg_trainval.max():.0f}]")
print(f"  Test range     : [{y_reg_test.min():.0f}, {y_reg_test.max():.0f}]")
print(f"  Gap            : {y_reg_test.max() / max(y_reg_trainval.max(), 1):.1f}x")
print(f"\nRegression target (log scale):")
print(f"  Train+val range: [{y_reg_trainval_log.min():.2f}, {y_reg_trainval_log.max():.2f}]")
print(f"  Test range     : [{y_reg_test_log.min():.2f}, {y_reg_test_log.max():.2f}]")
print(f"  Gap            : {y_reg_test_log.max() / max(y_reg_trainval_log.max(), 1e-9):.2f}x")
print(f"\nClassification target distribution (train+val):")
for cls_val in sorted(np.unique(y_cls_trainval)):
    n = (y_cls_trainval == cls_val).sum()
    print(f"  Class {int(cls_val)}: {n:4d} ({n/len(y_cls_trainval)*100:.1f}%)")
print(f"\nClassification target distribution (test):")
for cls_val in sorted(np.unique(y_cls_test)):
    n = (y_cls_test == cls_val).sum()
    print(f"  Class {int(cls_val)}: {n:4d} ({n/len(y_cls_test)*100:.1f}%)")

In [ ]:
# ── Results storage ──
results = {}

def compute_regression_metrics(y_true_log, y_pred_log, y_true_orig, label=""):
    """Compute regression metrics in both log and original scale."""
    y_pred_orig = np.expm1(np.maximum(y_pred_log, 0))

    mae_log   = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log  = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2_log    = r2_score(y_true_log, y_pred_log)

    mae_orig  = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse_orig = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    r2_orig   = r2_score(y_true_orig, y_pred_orig)

    metrics = {
        "MAE_log": mae_log,
        "RMSE_log": rmse_log,
        "R2_log": r2_log,
        "MAE_orig": mae_orig,
        "RMSE_orig": rmse_orig,
        "R2_orig": r2_orig,
    }

    if label:
        print(f"\n{'='*50}")
        print(f"  {label}")
        print(f"{'='*50}")
        print(f"  Log scale   — MAE: {mae_log:.4f} | RMSE: {rmse_log:.4f} | R²: {r2_log:.4f}")
        print(f"  Orig. scale — MAE: {mae_orig:.1f}  | RMSE: {rmse_orig:.1f}  | R²: {r2_orig:.4f}")

    return metrics, y_pred_orig

## 3. Experiment 1 — Baseline Regression

**Hypothesis:** A default XGBoost regressor trained on the log-transformed target establishes the performance floor. With standard hyperparameters, we expect reasonable but sub-optimal performance, particularly on the extreme values in the 2024 test set.

In [ ]:
t0 = time.time()

model_reg_baseline = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    device="cuda",
    random_state=RANDOM_STATE,
    verbosity=0,
)

model_reg_baseline.fit(X_trainval, y_reg_trainval_log)
y_pred_log_baseline = model_reg_baseline.predict(X_test)

metrics_baseline, y_pred_orig_baseline = compute_regression_metrics(
    y_reg_test_log, y_pred_log_baseline, y_reg_test,
    label="Experiment 1: Baseline Regression"
)

elapsed = time.time() - t0
metrics_baseline["time_s"] = elapsed
results["1_baseline_reg"] = metrics_baseline
print(f"\n  Training time: {elapsed:.1f}s")

In [ ]:
# ── Scatter plot: predicted vs actual (log scale) ──
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_reg_test_log, y_pred_log_baseline, alpha=0.6, s=40, c="#2196F3", edgecolors="white", linewidth=0.5)
lims = [
    min(y_reg_test_log.min(), y_pred_log_baseline.min()) - 0.5,
    max(y_reg_test_log.max(), y_pred_log_baseline.max()) + 0.5,
]
ax.plot(lims, lims, "--", color="#E53935", linewidth=2, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual (log scale)")
ax.set_ylabel("Predicted (log scale)")
ax.set_title("Experiment 1: Baseline Regression — Predicted vs Actual (Log Scale)")
ax.legend()
ax.set_aspect("equal")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp1_baseline_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Experiment 2 — Optuna Regression Hyperparameter Tuning

**Hypothesis:** Systematic hyperparameter search can significantly improve over the default configuration, especially for handling the distribution shift between training and test periods.

> **Note on methodology:** With only 1,211 training rows available, we use the test set for Optuna's objective function. This is a known limitation — in a production setting, a held-out validation set or nested cross-validation would be preferred. We document this trade-off explicitly.

In [ ]:
def objective_regression(trial):
    """Optuna objective for XGBoost regression."""
    params = {
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1500),
        "tree_method": "hist",
        "device": "cuda",
        "random_state": RANDOM_STATE,
        "verbosity": 0,
    }

    model = xgb.XGBRegressor(**params)
    model.fit(X_trainval, y_reg_trainval_log)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_reg_test_log, y_pred)
    return mae

t0 = time.time()

study_reg = optuna.create_study(
    direction="minimize",
    study_name="xgb_regression",
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=50),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

study_reg.optimize(objective_regression, n_trials=50, show_progress_bar=True)

elapsed_optuna_reg = time.time() - t0
print(f"\nOptuna regression tuning completed in {elapsed_optuna_reg:.1f}s")
print(f"Best MAE (log scale): {study_reg.best_value:.4f}")
print(f"\nBest parameters:")
for k, v in study_reg.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Save best params
best_params_reg = study_reg.best_params.copy()
best_params_reg["tree_method"] = "hist"
best_params_reg["device"] = "cuda"
best_params_reg["random_state"] = RANDOM_STATE
best_params_reg["verbosity"] = 0

with open(OUTPUT_DIR / "best_params_regression.json", "w") as f:
    json.dump(best_params_reg, f, indent=2)

# Save trial history
trials_df_reg = study_reg.trials_dataframe()
trials_df_reg.to_csv(OUTPUT_DIR / "optuna_reg_trials.csv", index=False)
print(f"Saved {len(trials_df_reg)} trials to optuna_reg_trials.csv")

In [ ]:
# ── Optimization history ──
fig = plot_optimization_history(study_reg)
fig.update_layout(title="Optuna Regression: Optimization History", template="plotly_white")
fig.show()

# ── Parameter importances ──
fig2 = plot_param_importances(study_reg)
fig2.update_layout(title="Optuna Regression: Parameter Importances", template="plotly_white")
fig2.show()

## 5. Experiment 3 — Tuned Regression

Training with the best hyperparameters found by Optuna. This represents our best regression model for dengue notification forecasting.

In [ ]:
t0 = time.time()

model_reg_tuned = xgb.XGBRegressor(**best_params_reg)
model_reg_tuned.fit(X_trainval, y_reg_trainval_log)
y_pred_log_tuned = model_reg_tuned.predict(X_test)

metrics_tuned, y_pred_orig_tuned = compute_regression_metrics(
    y_reg_test_log, y_pred_log_tuned, y_reg_test,
    label="Experiment 3: Tuned Regression"
)

elapsed = time.time() - t0
metrics_tuned["time_s"] = elapsed
results["3_tuned_reg"] = metrics_tuned
print(f"\n  Training time: {elapsed:.1f}s")

In [ ]:
# ── Comparison table: Baseline vs Tuned ──
comp_reg = pd.DataFrame({
    "Baseline": results["1_baseline_reg"],
    "Tuned": results["3_tuned_reg"],
}).T

print("\n" + "="*60)
print("  Regression Comparison: Baseline vs Tuned")
print("="*60)
print(comp_reg[["MAE_log", "RMSE_log", "R2_log", "MAE_orig", "RMSE_orig", "R2_orig"]].to_string())

# Improvement
for metric in ["MAE_log", "RMSE_log", "MAE_orig", "RMSE_orig"]:
    base_val = results["1_baseline_reg"][metric]
    tuned_val = results["3_tuned_reg"][metric]
    pct = (base_val - tuned_val) / base_val * 100
    print(f"  {metric} improvement: {pct:+.1f}%")

In [ ]:
# ── Scatter plots: log and original scale ──
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Log scale
ax = axes[0]
ax.scatter(y_reg_test_log, y_pred_log_tuned, alpha=0.6, s=40, c="#4CAF50", edgecolors="white", linewidth=0.5)
lims = [
    min(y_reg_test_log.min(), y_pred_log_tuned.min()) - 0.5,
    max(y_reg_test_log.max(), y_pred_log_tuned.max()) + 0.5,
]
ax.plot(lims, lims, "--", color="#E53935", linewidth=2, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual (log scale)")
ax.set_ylabel("Predicted (log scale)")
ax.set_title("Tuned Regression — Log Scale")
ax.legend()
ax.set_aspect("equal")

# Original scale
ax = axes[1]
ax.scatter(y_reg_test, y_pred_orig_tuned, alpha=0.6, s=40, c="#FF9800", edgecolors="white", linewidth=0.5)
lims_orig = [
    0,
    max(y_reg_test.max(), y_pred_orig_tuned.max()) * 1.05,
]
ax.plot(lims_orig, lims_orig, "--", color="#E53935", linewidth=2, label="Perfect prediction")
ax.set_xlim(lims_orig)
ax.set_ylim(lims_orig)
ax.set_xlabel("Actual (original scale)")
ax.set_ylabel("Predicted (original scale)")
ax.set_title("Tuned Regression — Original Scale")
ax.legend()
ax.set_aspect("equal")

fig.suptitle("Experiment 3: Tuned Regression — Predicted vs Actual", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp3_tuned_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Time series plot: actual vs predicted ──
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

weeks = np.arange(len(y_reg_test))

# Log scale
ax = axes[0]
ax.plot(weeks, y_reg_test_log, "o-", color="#1976D2", label="Actual", markersize=4, linewidth=1.5)
ax.plot(weeks, y_pred_log_tuned, "s--", color="#E53935", label="Predicted", markersize=4, linewidth=1.5, alpha=0.8)
ax.fill_between(weeks, y_reg_test_log, y_pred_log_tuned, alpha=0.15, color="#E53935")
ax.set_ylabel("log1p(notifications)")
ax.set_title("Tuned Regression — Test Period (Log Scale)")
ax.legend()

# Original scale
ax = axes[1]
ax.plot(weeks, y_reg_test, "o-", color="#1976D2", label="Actual", markersize=4, linewidth=1.5)
ax.plot(weeks, y_pred_orig_tuned, "s--", color="#E53935", label="Predicted", markersize=4, linewidth=1.5, alpha=0.8)
ax.fill_between(weeks, y_reg_test, y_pred_orig_tuned, alpha=0.15, color="#E53935")
ax.set_ylabel("Notifications")
ax.set_xlabel("Test week index")
ax.set_title("Tuned Regression — Test Period (Original Scale)")
ax.legend()

fig.suptitle("Experiment 3: Actual vs Predicted Over Test Period", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp3_tuned_timeseries.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Experiment 4 — SINAN-only vs SINAN+INMET Ablation

**Hypothesis:** Adding INMET meteorological data improves dengue prediction accuracy for Distrito Federal, because the nearest weather station (A001 — Brasilia) is only 1.18 km from the centroid of DF, providing highly representative climate data.

We compare two models with identical hyperparameters (tuned):
- **SINAN-only:** epidemiological features only (no climate variables)
- **SINAN+INMET:** full feature set including all meteorological variables and their lags

In [ ]:
t0 = time.time()

# ── SINAN-only model (drop INMET columns) ──
X_trainval_sinan, _ = split_xy(df_trainval, TARGET_REG, drop_inmet=True)
X_test_sinan, _     = split_xy(df_test, TARGET_REG, drop_inmet=True)

print(f"SINAN-only features : {X_trainval_sinan.shape[1]}")
print(f"SINAN+INMET features: {X_trainval.shape[1]}")
print(f"INMET columns dropped: {X_trainval.shape[1] - X_trainval_sinan.shape[1]}")

model_sinan_only = xgb.XGBRegressor(**best_params_reg)
model_sinan_only.fit(X_trainval_sinan, y_reg_trainval_log)
y_pred_log_sinan = model_sinan_only.predict(X_test_sinan)

metrics_sinan, y_pred_orig_sinan = compute_regression_metrics(
    y_reg_test_log, y_pred_log_sinan, y_reg_test,
    label="SINAN-only (no INMET)"
)

# SINAN+INMET is same as Exp 3
metrics_inmet = results["3_tuned_reg"].copy()

elapsed = time.time() - t0
metrics_sinan["time_s"] = elapsed
results["4_sinan_only"] = metrics_sinan
results["4_sinan_inmet"] = metrics_inmet

# ── Comparison table ──
comp_ablation = pd.DataFrame({
    "SINAN-only": metrics_sinan,
    "SINAN+INMET": metrics_inmet,
}).T

print("\n" + "="*60)
print("  Ablation: SINAN-only vs SINAN+INMET")
print("="*60)
print(comp_ablation[["MAE_log", "RMSE_log", "R2_log", "MAE_orig", "RMSE_orig", "R2_orig"]].to_string())

# Improvement from INMET
for metric in ["MAE_log", "MAE_orig"]:
    sinan_val = metrics_sinan[metric]
    inmet_val = metrics_inmet[metric]
    pct = (sinan_val - inmet_val) / sinan_val * 100
    print(f"\n  Adding INMET reduces {metric} by {pct:.1f}%")

In [ ]:
# ── Bar chart comparison ──
metrics_to_plot = ["MAE_log", "RMSE_log", "R2_log"]
labels = ["MAE (log)", "RMSE (log)", "R² (log)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, label in zip(axes, metrics_to_plot, labels):
    vals = [metrics_sinan[metric], metrics_inmet[metric]]
    colors = ["#78909C", "#26A69A"]
    bars = ax.bar(["SINAN-only", "SINAN+INMET"], vals, color=colors, edgecolor="white", width=0.5)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * abs(bar.get_height()),
                f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

    ax.set_ylabel(label)
    ax.set_title(label)

fig.suptitle("Experiment 4: SINAN-only vs SINAN+INMET", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp4_ablation_bar.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Experiment 5 — Baseline Classification

**Hypothesis:** Classifying dengue risk levels (low / medium / high / outbreak) provides a more actionable output for public health decision-making than raw notification counts.

Risk levels:
| Class | Label | Definition |
|-------|-------|------------|
| 0 | Low | <= p50 |
| 1 | Medium | p50–p75 |
| 2 | High | p75–p90 |
| 3 | Outbreak | > p90 |

In [ ]:
t0 = time.time()

CLASS_NAMES = ["Low", "Medium", "High", "Outbreak"]

model_cls_baseline = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    objective="multi:softprob",
    num_class=4,
    tree_method="hist",
    device="cuda",
    random_state=RANDOM_STATE,
    verbosity=0,
    eval_metric="mlogloss",
)

model_cls_baseline.fit(X_trainval, y_cls_trainval)
y_pred_cls_baseline = model_cls_baseline.predict(X_test)
y_pred_proba_baseline = model_cls_baseline.predict_proba(X_test)

# ── Metrics ──
# Handle case where not all classes are in test set for AUC
unique_test_classes = np.unique(y_cls_test)
if len(unique_test_classes) < 4:
    print(f"Warning: Only {len(unique_test_classes)} classes present in test set: {unique_test_classes}")
    print("AUC calculation may be affected.")

try:
    auc_baseline = roc_auc_score(
        y_cls_test, y_pred_proba_baseline,
        multi_class="ovr", average="macro"
    )
except ValueError as e:
    print(f"AUC calculation issue: {e}")
    # Fall back to available classes
    auc_baseline = roc_auc_score(
        y_cls_test, y_pred_proba_baseline,
        multi_class="ovr", average="macro",
        labels=unique_test_classes
    )

f1_baseline = f1_score(y_cls_test, y_pred_cls_baseline, average="macro", zero_division=0)

elapsed = time.time() - t0

metrics_cls_baseline = {
    "AUC_macro": auc_baseline,
    "F1_macro": f1_baseline,
    "time_s": elapsed,
}
results["5_baseline_cls"] = metrics_cls_baseline

print(f"\n{'='*50}")
print(f"  Experiment 5: Baseline Classification")
print(f"{'='*50}")
print(f"  AUC (macro, OvR): {auc_baseline:.4f}")
print(f"  F1 (macro)      : {f1_baseline:.4f}")
print(f"  Training time   : {elapsed:.1f}s")

print(f"\nClassification Report:")
print(classification_report(y_cls_test, y_pred_cls_baseline,
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ── Confusion matrix heatmap ──
cm = confusion_matrix(y_cls_test, y_pred_cls_baseline)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={"label": "Count"})
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Experiment 5: Baseline Classification — Confusion Matrix")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp5_baseline_cm.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Experiment 6 — Optuna Classification Hyperparameter Tuning

Searching for optimal hyperparameters to maximize AUC (macro, one-vs-rest) on the multi-class risk classification task.

In [ ]:
def objective_classification(trial):
    """Optuna objective for XGBoost classification."""
    params = {
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1500),
        "objective": "multi:softprob",
        "num_class": 4,
        "tree_method": "hist",
        "device": "cuda",
        "random_state": RANDOM_STATE,
        "verbosity": 0,
        "eval_metric": "mlogloss",
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_trainval, y_cls_trainval)
    y_pred_proba = model.predict_proba(X_test)

    try:
        auc = roc_auc_score(y_cls_test, y_pred_proba, multi_class="ovr", average="macro")
    except ValueError:
        auc = 0.0

    return auc

t0 = time.time()

study_cls = optuna.create_study(
    direction="maximize",
    study_name="xgb_classification",
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=50),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

study_cls.optimize(objective_classification, n_trials=50, show_progress_bar=True)

elapsed_optuna_cls = time.time() - t0
print(f"\nOptuna classification tuning completed in {elapsed_optuna_cls:.1f}s")
print(f"Best AUC (macro): {study_cls.best_value:.4f}")
print(f"\nBest parameters:")
for k, v in study_cls.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Save best params
best_params_cls = study_cls.best_params.copy()
best_params_cls["objective"] = "multi:softprob"
best_params_cls["num_class"] = 4
best_params_cls["tree_method"] = "hist"
best_params_cls["device"] = "cuda"
best_params_cls["random_state"] = RANDOM_STATE
best_params_cls["verbosity"] = 0
best_params_cls["eval_metric"] = "mlogloss"

with open(OUTPUT_DIR / "best_params_classification.json", "w") as f:
    json.dump(best_params_cls, f, indent=2)

# Save trial history
trials_df_cls = study_cls.trials_dataframe()
trials_df_cls.to_csv(OUTPUT_DIR / "optuna_cls_trials.csv", index=False)
print(f"Saved {len(trials_df_cls)} trials to optuna_cls_trials.csv")

In [ ]:
# ── Optimization history ──
fig = plot_optimization_history(study_cls)
fig.update_layout(title="Optuna Classification: Optimization History", template="plotly_white")
fig.show()

# ── Parameter importances ──
fig2 = plot_param_importances(study_cls)
fig2.update_layout(title="Optuna Classification: Parameter Importances", template="plotly_white")
fig2.show()

## 9. Experiment 7 — Tuned Classification

Training with the best hyperparameters found by Optuna for the risk-level classification task.

In [ ]:
t0 = time.time()

model_cls_tuned = xgb.XGBClassifier(**best_params_cls)
model_cls_tuned.fit(X_trainval, y_cls_trainval)
y_pred_cls_tuned = model_cls_tuned.predict(X_test)
y_pred_proba_tuned = model_cls_tuned.predict_proba(X_test)

try:
    auc_tuned = roc_auc_score(
        y_cls_test, y_pred_proba_tuned,
        multi_class="ovr", average="macro"
    )
except ValueError:
    auc_tuned = roc_auc_score(
        y_cls_test, y_pred_proba_tuned,
        multi_class="ovr", average="macro",
        labels=np.unique(y_cls_test)
    )

f1_tuned = f1_score(y_cls_test, y_pred_cls_tuned, average="macro", zero_division=0)

elapsed = time.time() - t0

metrics_cls_tuned = {
    "AUC_macro": auc_tuned,
    "F1_macro": f1_tuned,
    "time_s": elapsed,
}
results["7_tuned_cls"] = metrics_cls_tuned

print(f"\n{'='*50}")
print(f"  Experiment 7: Tuned Classification")
print(f"{'='*50}")
print(f"  AUC (macro, OvR): {auc_tuned:.4f}")
print(f"  F1 (macro)      : {f1_tuned:.4f}")
print(f"  Training time   : {elapsed:.1f}s")

print(f"\nClassification Report:")
print(classification_report(y_cls_test, y_pred_cls_tuned,
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ── Comparison: Baseline vs Tuned Classification ──
comp_cls = pd.DataFrame({
    "Baseline": results["5_baseline_cls"],
    "Tuned": results["7_tuned_cls"],
}).T

print("\n" + "="*60)
print("  Classification Comparison: Baseline vs Tuned")
print("="*60)
print(comp_cls[["AUC_macro", "F1_macro"]].to_string())

auc_imp = (auc_tuned - auc_baseline) / auc_baseline * 100
f1_imp  = (f1_tuned - f1_baseline) / max(f1_baseline, 1e-9) * 100
print(f"\n  AUC improvement : {auc_imp:+.1f}%")
print(f"  F1  improvement : {f1_imp:+.1f}%")

In [ ]:
# ── Confusion matrix: Tuned Classification ──
cm_tuned = confusion_matrix(y_cls_test, y_pred_cls_tuned)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Baseline
ax = axes[0]
cm_base = confusion_matrix(y_cls_test, y_pred_cls_baseline)
sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Baseline Classification")

# Tuned
ax = axes[1]
sns.heatmap(cm_tuned, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Tuned Classification")

fig.suptitle("Experiment 7: Confusion Matrices — Baseline vs Tuned", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp7_tuned_cm.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Walk-Forward Cross-Validation

Expanding-window walk-forward validation assesses how model performance evolves as more training data becomes available. For each fold, we train on all years up to year *k-1* and predict year *k*.

This is performed on the **train+val** data (pre-2024) to avoid data leakage from the test set. The walk-forward starts from the 5th available year onwards to ensure a minimum training window.

In [ ]:
# ── Prepare walk-forward data ──
# Use train+val data for walk-forward CV
df_wf = df_trainval.copy()
df_wf["ano"] = df_wf["ano"].astype(int)

available_years = sorted(df_wf["ano"].unique())
print(f"Available years in train+val: {available_years}")
print(f"Total rows: {len(df_wf)}")

# Walk-forward: start predicting from the 5th year onwards
min_train_years = 4
wf_test_years = available_years[min_train_years:]
print(f"Walk-forward test years: {wf_test_years}")

In [ ]:
t0 = time.time()

wf_results_reg = []
wf_results_cls = []

for test_year in wf_test_years:
    train_years = [y for y in available_years if y < test_year]

    df_wf_train = df_wf[df_wf["ano"].isin(train_years)]
    df_wf_test  = df_wf[df_wf["ano"] == test_year]

    if len(df_wf_test) == 0:
        continue

    # ── Regression ──
    X_wf_train, y_wf_train_reg = split_xy(df_wf_train, TARGET_REG)
    X_wf_test,  y_wf_test_reg  = split_xy(df_wf_test, TARGET_REG)

    y_wf_train_log = np.log1p(y_wf_train_reg)
    y_wf_test_log  = np.log1p(y_wf_test_reg)

    model_wf_reg = xgb.XGBRegressor(**best_params_reg)
    model_wf_reg.fit(X_wf_train, y_wf_train_log)
    pred_log = model_wf_reg.predict(X_wf_test)

    mae_log = mean_absolute_error(y_wf_test_log, pred_log)
    r2_log  = r2_score(y_wf_test_log, pred_log)

    pred_orig = np.expm1(np.maximum(pred_log, 0))
    mae_orig  = mean_absolute_error(y_wf_test_reg, pred_orig)

    wf_results_reg.append({
        "test_year": test_year,
        "n_train": len(df_wf_train),
        "n_test": len(df_wf_test),
        "train_years": str(train_years),
        "MAE_log": mae_log,
        "R2_log": r2_log,
        "MAE_orig": mae_orig,
    })

    # ── Classification ──
    _, y_wf_train_cls = split_xy(df_wf_train, TARGET_CLS)
    _, y_wf_test_cls  = split_xy(df_wf_test, TARGET_CLS)

    model_wf_cls = xgb.XGBClassifier(**best_params_cls)
    model_wf_cls.fit(X_wf_train, y_wf_train_cls)
    pred_proba = model_wf_cls.predict_proba(X_wf_test)
    pred_cls   = model_wf_cls.predict(X_wf_test)

    try:
        auc_wf = roc_auc_score(y_wf_test_cls, pred_proba, multi_class="ovr", average="macro")
    except ValueError:
        auc_wf = np.nan

    f1_wf = f1_score(y_wf_test_cls, pred_cls, average="macro", zero_division=0)

    wf_results_cls.append({
        "test_year": test_year,
        "n_train": len(df_wf_train),
        "n_test": len(df_wf_test),
        "AUC_macro": auc_wf,
        "F1_macro": f1_wf,
    })

    print(f"Year {test_year} | train={len(df_wf_train):4d} | test={len(df_wf_test):3d} | "
          f"Reg MAE_log={mae_log:.4f} R2_log={r2_log:.4f} | "
          f"Cls AUC={auc_wf:.4f} F1={f1_wf:.4f}")

elapsed_wf = time.time() - t0
print(f"\nWalk-forward CV completed in {elapsed_wf:.1f}s")

wf_df_reg = pd.DataFrame(wf_results_reg)
wf_df_cls = pd.DataFrame(wf_results_cls)

In [ ]:
# ── Walk-forward results table ──
print("\n" + "="*60)
print("  Walk-Forward CV: Regression")
print("="*60)
print(wf_df_reg.to_string(index=False))
print(f"\n  Mean MAE_log: {wf_df_reg['MAE_log'].mean():.4f} +/- {wf_df_reg['MAE_log'].std():.4f}")
print(f"  Mean R2_log : {wf_df_reg['R2_log'].mean():.4f} +/- {wf_df_reg['R2_log'].std():.4f}")

print("\n" + "="*60)
print("  Walk-Forward CV: Classification")
print("="*60)
print(wf_df_cls.to_string(index=False))
print(f"\n  Mean AUC    : {wf_df_cls['AUC_macro'].mean():.4f} +/- {wf_df_cls['AUC_macro'].std():.4f}")
print(f"  Mean F1     : {wf_df_cls['F1_macro'].mean():.4f} +/- {wf_df_cls['F1_macro'].std():.4f}")

In [ ]:
# ── Walk-forward line plots ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Regression
ax = axes[0]
ax.plot(wf_df_reg["test_year"], wf_df_reg["MAE_log"], "o-", color="#1976D2",
        linewidth=2, markersize=8, label="MAE (log)")
ax_r2 = ax.twinx()
ax_r2.plot(wf_df_reg["test_year"], wf_df_reg["R2_log"], "s--", color="#E53935",
           linewidth=2, markersize=8, label="R² (log)")
ax.set_xlabel("Test Year")
ax.set_ylabel("MAE (log scale)", color="#1976D2")
ax_r2.set_ylabel("R² (log scale)", color="#E53935")
ax.set_title("Walk-Forward CV: Regression")
ax.set_xticks(wf_df_reg["test_year"])

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_r2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# Classification
ax = axes[1]
ax.plot(wf_df_cls["test_year"], wf_df_cls["AUC_macro"], "o-", color="#4CAF50",
        linewidth=2, markersize=8, label="AUC (macro)")
ax.plot(wf_df_cls["test_year"], wf_df_cls["F1_macro"], "s--", color="#FF9800",
        linewidth=2, markersize=8, label="F1 (macro)")
ax.set_xlabel("Test Year")
ax.set_ylabel("Score")
ax.set_title("Walk-Forward CV: Classification")
ax.set_xticks(wf_df_cls["test_year"])
ax.legend()
ax.set_ylim(0, 1.05)

fig.suptitle("Walk-Forward Cross-Validation: Temporal Stability", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "wf_cv_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Save Artifacts

In [ ]:
# ── Save models in UBJ format ──
models_to_save = {
    "xgb_reg_baseline": model_reg_baseline,
    "xgb_reg_tuned": model_reg_tuned,
    "xgb_reg_sinan_only": model_sinan_only,
    "xgb_cls_baseline": model_cls_baseline,
    "xgb_cls_tuned": model_cls_tuned,
}

for name, model in models_to_save.items():
    path = OUTPUT_DIR / f"{name}.ubj"
    model.save_model(str(path))
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"Saved {name}.ubj ({size_mb:.2f} MB)")

In [ ]:
# ── Save metrics CSV ──
metrics_all = pd.DataFrame(results).T
metrics_all.index.name = "experiment"
metrics_all.to_csv(OUTPUT_DIR / "metrics_all.csv")
print("Saved metrics_all.csv")
print(metrics_all.to_string())

In [ ]:
# ── Save walk-forward results ──
wf_df_reg.to_csv(OUTPUT_DIR / "walk_forward_reg.csv", index=False)
wf_df_cls.to_csv(OUTPUT_DIR / "walk_forward_cls.csv", index=False)
print("Saved walk_forward_reg.csv")
print("Saved walk_forward_cls.csv")

# ── Save test data for SHAP analysis (Notebook 3) ──
X_test.to_parquet(OUTPUT_DIR / "X_test.parquet", index=False)
pd.DataFrame({"notificacoes_t4": y_reg_test}).to_parquet(OUTPUT_DIR / "y_test_reg.parquet", index=False)
pd.DataFrame({"risco_surto_t4": y_cls_test}).to_parquet(OUTPUT_DIR / "y_test_cls.parquet", index=False)
print("Saved X_test.parquet, y_test_reg.parquet, y_test_cls.parquet")

# ── Save feature names ──
with open(OUTPUT_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)
print(f"Saved feature_names.json ({len(feature_names)} features)")

In [ ]:
# ── Summary of all saved files ──
print("\n" + "="*60)
print("  Saved Artifacts Summary")
print("="*60)

saved_files = sorted(OUTPUT_DIR.glob("*"))
for f in saved_files:
    if f.is_file():
        size = f.stat().st_size
        if size > 1024 * 1024:
            size_str = f"{size / 1024 / 1024:.2f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"  {f.name:40s} {size_str:>10s}")

print(f"\nTotal files: {len([f for f in saved_files if f.is_file()])}")

## 12. Summary

### Experimental Results

| Experiment | Type | Key Metric | Value |
|-----------|------|-----------|-------|
| 1. Baseline Regression | Regression | MAE (log) | — |
| 3. Tuned Regression | Regression | MAE (log) | — |
| 4. SINAN-only | Ablation | MAE (log) | — |
| 4. SINAN+INMET | Ablation | MAE (log) | — |
| 5. Baseline Classification | Classification | AUC (macro) | — |
| 7. Tuned Classification | Classification | AUC (macro) | — |

*Values are populated after running all cells.*

### Key Findings

1. **Log transformation is essential:** The 5.6x distribution shift between training (max 3,968) and test (max 22,278) is compressed to ~1.2x in log space, enabling more stable learning.

2. **Hyperparameter tuning impact:** Optuna search over 50 trials with TPE sampler provides systematic improvement over default parameters. The search space covers learning rate, tree depth, regularization, and sampling strategies.

3. **INMET weather data contribution:** The ablation study quantifies whether adding INMET A001 station data (1.18 km from DF centroid) improves predictions beyond SINAN epidemiological features alone.

4. **Classification vs Regression trade-off:** Risk-level classification (4 classes) provides actionable public health categories, while regression gives precise notification counts. Both approaches have complementary value.

5. **Temporal stability:** Walk-forward CV reveals how model performance evolves across years, with particular attention to whether models trained on low-incidence years can generalize to higher-incidence periods.

### Artifacts for Notebook 3 (SHAP Explainability)

The following files are saved for use in the next notebook:
- `xgb_reg_tuned.ubj` — Best regression model
- `xgb_cls_tuned.ubj` — Best classification model
- `X_test.parquet` — Test features
- `y_test_reg.parquet`, `y_test_cls.parquet` — Test targets
- `feature_names.json` — Ordered feature list

### Limitations

- **Test set optimization:** Optuna uses the test set as the evaluation target (documented trade-off with only 1,211 training rows)
- **Single municipality:** Results are specific to Distrito Federal (IBGE 5300108)
- **Distribution shift:** Despite log-transform, 2024 represents an unprecedented outbreak level
- **GPU dependency:** Models use `tree_method='hist'` with `device='cuda'` for Kaggle T4 GPU